In [27]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss


from datasets import load_dataset

In [28]:
ds = load_dataset("higopires/RePro-categories-multilabel")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_text             1007 non-null   object
 1   ENTREGA                 1007 non-null   int64 
 2   OUTROS                  1007 non-null   int64 
 3   PRODUTO                 1007 non-null   int64 
 4   CONDICOESDERECEBIMENTO  1007 non-null   int64 
 5   INADEQUADA              1007 non-null   int64 
 6   ANUNCIO                 1007 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 55.2+ KB


In [29]:
test = test[test['INADEQUADA'] == 0].reset_index(drop=True)

test = test.drop(columns=['INADEQUADA'])

test.rename(columns={'review_text': 'text'}, inplace=True)

test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0
3,bom..............................................,0,0,1,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0
...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1


In [30]:
test.rename(columns={'CONDICOESDERECEBIMENTO': 'CONDICOES DE RECEBIMENTO'}, inplace=True)

labels = test.columns[1:]

test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOES DE RECEBIMENTO,ANUNCIO
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0
3,bom..............................................,0,0,1,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0
...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1


In [31]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [32]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "Você é um assistente de classificação. Seu objetivo é ler o texto fornecido e classificá-lo de acordo com a tarefa e os rótulos descritos. Você é capaz de lidar com tarefas de classificação multilabel com base nas instruções do user."},
            {"role": "user", "content": f"Classifique o seguinte texto com base na tarefa: Análise de categorias de avaliações de produtos e-commerce. Responda apenas com os rótulos que melhor descrevem o texto. Se houver mais de um rótulo, os separe com vírgula. Os rótulos possíveis são: {', '.join(labels)}. Texto: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    list = []
    if 'entrega' in text:
        list.append('ENTREGA')
    if 'produtos' in text:
        list.append('PRODUTOS')
    if 'condicoes de recebimento' in text:
        list.append('CONDICOES DE RECEBIMENTO')
    if 'anuncio' in text:
        list.append('ANUNCIO')
    if 'outros' in text:
        list.append('OUTROS')

    return list

In [33]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_multilabel1.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_multilabel1.csv", index=False)

pred_df

,text,ENTREGA,OUTROS,PRODUTO,CONDICOES DE RECEBIMENTO,ANUNCIO,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0,"entrega, produto, outros",1.064493,10.0,336.0,346.0,"[ENTREGA, OUTROS]"
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0,"produto, outros",0.541386,7.0,160.0,167.0,[OUTROS]
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0,"produto, condicoes de recebimento",0.749639,13.0,165.0,178.0,[CONDICOES DE RECEBIMENTO]
3,bom..............................................,0,0,1,0,0,outros,0.682063,3.0,152.0,155.0,[OUTROS]
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0,"entrega, outros",1.192835,7.0,158.0,165.0,"[ENTREGA, OUTROS]"
...,...,...,...,...,...,...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1,"produto, anuncio",0.828525,8.0,158.0,166.0,[ANUNCIO]
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1,"produto, anuncio, outros",0.560511,11.0,167.0,178.0,"[ANUNCIO, OUTROS]"
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1,"anuncio, entrega, outros",0.483532,10.0,178.0,188.0,"[ENTREGA, ANUNCIO, OUTROS]"
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1,"produto, anuncio, condicoes de recebimento",0.765037,17.0,172.0,189.0,"[CONDICOES DE RECEBIMENTO, ANUNCIO]"


In [34]:
for label in labels:
    pred_df[f"{label} pred"] = pred_df.apply(lambda row: 1 if label in row['prediction_post_processed'] else 0, axis=1)

pred_df = pred_df.drop(columns=['prediction'])

pred_df

,text,ENTREGA,OUTROS,PRODUTO,CONDICOES DE RECEBIMENTO,ANUNCIO,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed,ENTREGA pred,OUTROS pred,PRODUTO pred,CONDICOES DE RECEBIMENTO pred,ANUNCIO pred
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0,1.064493,10.0,336.0,346.0,"[ENTREGA, OUTROS]",1,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0,0.541386,7.0,160.0,167.0,[OUTROS],0,1,0,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0,0.749639,13.0,165.0,178.0,[CONDICOES DE RECEBIMENTO],0,0,0,1,0
3,bom..............................................,0,0,1,0,0,0.682063,3.0,152.0,155.0,[OUTROS],0,1,0,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0,1.192835,7.0,158.0,165.0,"[ENTREGA, OUTROS]",1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1,0.828525,8.0,158.0,166.0,[ANUNCIO],0,0,0,0,1
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1,0.560511,11.0,167.0,178.0,"[ANUNCIO, OUTROS]",0,1,0,0,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1,0.483532,10.0,178.0,188.0,"[ENTREGA, ANUNCIO, OUTROS]",1,1,0,0,1
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1,0.765037,17.0,172.0,189.0,"[CONDICOES DE RECEBIMENTO, ANUNCIO]",0,0,0,1,1


In [35]:
y_true = pred_df[labels].values
y_pred = pred_df[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.071429
F1 score: 0.322732
Precision: 0.319596
Recall: 0.370249
Hamming loss: 0.327743


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.6865527128827744
Average completion tokens: 8.261904761904763
Average prompt tokens: 181.296066252588
Average total tokens: 189.55797101449275


In [37]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.03105840000000003


In [38]:
with open('results/openai_ZS_multilabel1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Hamming loss: {hamming_loss}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')